![NVIDIA Logo](images/nvidia.png)

# Zip Nodes

In this notebook we introduce the MRC node `ZipComponent` which can be used inside of custom stages to create nodes capable of combining data from multiple input ports. This combination strategy waits for messages to arrive from each upstream branch before combining. Scenarios where this can be useful is when each branch is performing parallel operations that need to be recombined before the next stage of processing can begin.

---

## Objectives

By the time you complete this notebook you will be able to:

- Create custom stages capable of zipping multiple incoming messages.

---

## Imports

In [ ]:
from typing import List
import typing
import logging

from IPython.display import Image
import cudf

from morpheus.pipeline import Stage, Pipeline
from morpheus.pipeline.stage_schema import StageSchema
from morpheus.config import Config
from morpheus.pipeline.execution_mode_mixins import GpuAndCpuMixin
from morpheus.messages import MessageMeta

from morpheus.stages.general.monitor_stage import MonitorStage
from morpheus.stages.input.file_source_stage import FileSourceStage
from morpheus.stages.output.in_memory_sink_stage import InMemorySinkStage
from morpheus.utils.logger import configure_logging, reset_logging

import mrc
import mrc.core.operators as ops
from mrc.core.node import ZipComponent

import cudf

---

## Concatenate Messages

Later in the notebook we will be creating a custom stage that can zip two messages together. When creating our zip component, we will need to provide a function that defines how the two messages ought to be zipped.

`concatenate_messages` expects a list of `MessageMeta` messages and, in a thread-safe manner, returns a new message that is a concatenation of those passed in.

In [ ]:
def concatenate_messages(messages: List[MessageMeta]) -> MessageMeta:
    """
    Thread-safe function that concatenates multiple MessageMeta objects into one.

    Args:
        messages (List[MessageMeta]): A list of MessageMeta objects to concatenate.

    Returns:
        MessageMeta: A new MessageMeta object containing the data concatenated along the column axis.
    """
    if not messages:
        raise ValueError("No messages provided for concatenation.")

    # Extract DataFrames in a thread-safe way
    dataframes = []
    for idx, message in enumerate(messages):
        df = message.copy_dataframe()
        df.columns = [f"{col_name}_{idx}" for col_name in df.columns]
        dataframes.append(df)

    # Concatenate DataFrames along column axis
    combined_df = cudf.concat(dataframes, axis=1)
    

    return MessageMeta(combined_df)


In [ ]:
input_file = "data/simple_user_log.jsonlines"

In [ ]:
df = cudf.read_json(input_file, lines=True)

In [ ]:
mm = MessageMeta(df)

In [ ]:
mm2 = MessageMeta(df)

In [ ]:
concatenate_messages(messages=[mm, mm2]).get_data()

---

## Zip Component

We can use `mrc.core.nodes.ZipComponent` in custom stages to create a zip node. It will expect a number of inputs specified by its `count` argument, and will pass its inputs as a list to its `convert_fn` argument, which should expect a list of messages (like our `concatenate_messages` function above).

```python
zip_component = ZipComponent(builder=builder, name="zip", count=num_inputs, convert_fn=join_messages)
```

---

## Custom Zip Stage

Here we define a custom `ZipStage`. To make it re-usable we've parameterized the number of inputs it can receive and also the `convert_function` it will use to zip incoming messages.

When constructing edges from multiple inputs into a single zip component, we use the zip components `get_sink` method, along with an index, to specify which input port the edge should be connected to.

In [ ]:
class ZipStage(GpuAndCpuMixin, Stage):

    def __init__(self, c: Config, num_inputs: int, convert_fn, convert_args: dict = None):
        """
        A custom stage that zips multiple input messages into one.

        Args:
            c (Config): The Morpheus pipeline configuration.
            num_inputs (int): Number of input ports (messages to zip).
            convert_fn (callable): Function that merges multiple MessageMeta objects.
            convert_args (dict, optional): Additional arguments to pass to `convert_fn`.
        """
        super().__init__(c)

        self._num_inputs = num_inputs
        self._convert_fn = convert_fn
        self._convert_args = convert_args or {}  # Default to empty dict

        self._create_ports(self._num_inputs, 1)  # Multiple input ports, one output port

    @property
    def name(self) -> str:
        return "mp-zip"

    def supports_cpp_node(self):
        return False

    # This stage only has a single output port, so we only need to define the output schema
    # for a single output.
    def compute_schema(self, schema: StageSchema):
        schema.output_schemas[0].set_type(typing.Any)

    def on_data(self, message: typing.Any) -> typing.Any:
        return message

    def _build(self, builder: mrc.Builder, input_nodes: list[mrc.SegmentObject]) -> list[mrc.SegmentObject]:
        """
        Defines the pipeline structure.
        """

        def thread_safe_join(messages: List[MessageMeta]) -> MessageMeta:
            """
            Calls the user-provided `convert_fn` while ensuring thread safety.

            Args:
                messages (List[MessageMeta]): A list of input messages.

            Returns:
                MessageMeta: The processed MessageMeta object.
            """
            return self._convert_fn(messages, **self._convert_args)

        # Create ZipComponent using the parameterized conversion function
        zip_component = ZipComponent(
            builder=builder,
            name=f"zip-{self.unique_name}",
            count=self._num_inputs,
            convert_fn=thread_safe_join
        )

        # Connect all input nodes to the zip component
        for i, input_node in enumerate(input_nodes):
            # We use the `get_sink` method to connect incoming edges to various input ports.
            builder.make_edge(input_node, zip_component.get_sink(i))

        return [zip_component]

---

## Zip Multiple Inputs in a Pipeline

In [ ]:
input_file = 'data/simple_user_log.jsonlines'

In [ ]:
config = Config()
pipeline = Pipeline(config)

source_1 = pipeline.add_stage(FileSourceStage(config, filename=input_file, iterative=False, repeat=100))
source_2 = pipeline.add_stage(FileSourceStage(config, filename=input_file, iterative=False, repeat=100))

zip_stage = pipeline.add_stage(ZipStage(config, num_inputs=2, convert_fn=concatenate_messages))

pipeline.add_edge(source_1, zip_stage.input_ports[0])
pipeline.add_edge(source_2, zip_stage.input_ports[1])

in_mem_sink = pipeline.add_stage(InMemorySinkStage(config))
monitor = pipeline.add_stage(MonitorStage(config, description="Pipeline throughput"))

pipeline.add_edge(zip_stage, in_mem_sink)
pipeline.add_edge(in_mem_sink, monitor)

In [ ]:
pipeline.build()

In [ ]:
viz_file = 'pipeline_visualizations/zip_passthrough.png'
pipeline.visualize(viz_file)

In [ ]:
Image(filename=viz_file)

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

In [ ]:
await pipeline.run_async()

In [ ]:
messages = in_mem_sink.get_messages()
messages[0].get_data()